# Preprocesamiento de Texto

Notebook de la etapa **0** del pipeline de detección de cyberbullying en texto.
Aquí se define y se aplica la función de normalización `normalize()` que prepara
el texto crudo de los tweets antes de la extracción de características (01), el
análisis exploratorio (02) y el modelado (03).

## Pipeline de normalización

La función `normalize()` aplica, en orden estricto:

1. **Minúsculas**: `text.lower()`.
2. **Entidades HTML**: `html.unescape()` decodifica `&amp;`, `&lt;`, etc.
3. **Expansión de contracciones** (`CONTRACTIONS`): `don't` → `do not`.
4. **Normalización de slang y leetspeak** (`SLANG` / `LEET`): `u` → `you`,
   `b4` → `before`.
5. **Limpieza regex**: se eliminan caracteres no-ASCII, emails, URLs,
   menciones, hashtags, la marca `RT` y todo lo que no sea letra `a-zA-Z`.
6. **Unicode**: `unidecode` translitera acentos y caracteres extendidos.
7. **Elongación**: `sooooo` → `so`, `noooo` → `no`.
8. **Filtrado léxico con `tokenizer`** (pipeline mínimo): se descartan
   stopwords, signos de puntuación y números, tokens de longitud ≤ 2 y tokens
   con un único carácter distinto (p. ej. `zzzz`).
9. **Lematización con `lemmatizer` completo** (tagger + parser): cada palabra se
   reduce a su lema con contexto sintáctico.
10. **Marcado de negación**: `not good` → `not_good`.

El **orden no es arbitrario**: cada paso condiciona al siguiente. Por ejemplo,
la expansión de contracciones debe ocurrir antes de la limpieza regex para que
el `not` de `doesn't` sobreviva y pueda disparar el marcado de negación; y la
negación se marca sobre los lemas, no antes, para no romper la lematización.
Las justificaciones de cada decisión se explican en las celdas correspondientes
y se resumen en la sección final "Decisiones de normalización y justificación".

El objetivo es reducir el ruido típico de las redes sociales y unificar las
variantes de una misma palabra antes de la vectorización.


## Importación de librerías

Se importan las librerías de manipulación de datos (`pandas`), expresiones
regulares (`re`), entidades HTML (`html`) y transliteración Unicode
(`unidecode`). `tqdm.pandas()` habilita las barras de progreso sobre operaciones
vectoriales.

En la celda siguiente también se cargan los modelos de spaCy que usa
`normalize()`; la justificación de los dos pipelines (completo y mínimo) se
detalla en la sección siguiente.


## Carga de los modelos de spaCy

`normalize()` necesita **dos pipelines de spaCy distintos** sobre el mismo modelo
`en_core_web_sm`. La diferencia es una decisión deliberada de rendimiento:

- **`lemmatizer`** (modelo completo: tagger, parser, attribute_ruler y
  lemmatizer): se usa **solo** para la lematización. El parser influye en el lema
  elegido: por ejemplo, `going` lematiza como `going` con parser y como `go` sin
  él, porque el contexto sintáctico cambia la interpretación de la palabra. Por
  eso la lematización no se delega en un pipeline mínimo.
- **`tokenizer`** (pipeline mínimo: solo tokenizer + atributos léxicos): se usa
  para los filtros de `keep_token` (`is_stop`, `is_punct`, `like_num`, longitud).
  Estos atributos no necesitan tagger ni parser, por lo que desactivarlos hace el
  filtrado **~8 veces más rápido** sin perder precisión.

En caso de no tener instalado el modelo, se instala con
`python -m spacy download en_core_web_sm` (para español: `es_core_news_sm`).


In [1]:
import html
import re

import pandas as pd
import spacy
from tqdm import tqdm
from unidecode import unidecode

tqdm.pandas()

lemmatizer = spacy.load('en_core_web_sm', disable=['ner'])

tokenizer = spacy.load('en_core_web_sm', disable=['tagger', 'parser', 'ner', 'attribute_ruler', 'lemmatizer'])

## Normalización de texto

`normalize()` encadena la limpieza (función `clean`) con el filtrado léxico, la
lematización y el marcado de negación. A continuación se documenta el porqué de
cada paso y el orden en que se aplican; el orden no es casual: cada
transformación prepara el texto para la siguiente.

### Orden de procesamiento

1. **Expansión de contracciones, slang y leetspeak** (antes de la limpieza
   regex): `doesn't` → `does not` deja visible el `not` que luego dispara el
   marcado de negación. Si la limpieza regex ocurriera primero, el apóstrofo se
   eliminaría y la negación se perdería.
2. **Limpieza regex y Unicode**: se eliminan no-ASCII, emails, URLs, menciones,
   hashtags, `RT` y todo lo que no sea `a-zA-Z`; luego `unidecode` unifica
   acentos y caracteres extendidos.
3. **Elongación**: se contraen las repeticiones tipográficas (`sooooo` → `so`),
   evitando que variantes de la misma palabra generen tokens distintos.
4. **Filtrado léxico** (`keep_token` con `tokenizer`): se descartan stopwords,
   signos de puntuación y números, tokens de ≤ 2 caracteres y tokens con un único
   carácter distinto (`zzzz`).
5. **Lematización con `lemmatizer` completo**: cada token se reduce a su lema usando el
   contexto del parser.
6. **Marcado de negación sobre los lemas**: `not good` → `not_good`.

### Por qué la negación va DESPUÉS de la lematización

Si se marcara la negación sobre tokens sin lematizar, `not_going` se lematizaría
luego como `not_goe` (el prefijo `not_` rompe el lema del verbo). Al lematizar
primero y marcar la negación al final, el prefijo se aplica sobre la forma
canónica. Una negación huérfana al final (p. ej. `I do not`) se conserva como
`not` para no perderla.

### Por qué las negaciones se conservan aunque sean stopwords

`not`, `no` y `never` son stopwords de spaCy. El filtro `keep_token` las conserva
de forma explícita (conjunto `NEGATION_WORDS`): si se descartaran junto con el
resto de las stopwords, la negación desaparecería y el texto perdería
una señal importante para la detección de cyberbullying.


### Diccionarios de expansión: contracciones, slang y leetspeak

- **`CONTRACTIONS`**: contracciones inglesas estándar (`don't` → `do not`). La
  expansión es clave para la negación: `don't talk` → `do not talk` → `not_talk`.
- **`SLANG`**: formas coloquiales de redes sociales (`u` → `you`,
  `idk` → `i do not know`). Varias de ellas alimentan la negación
  (`aint` → `are not`).
- **`LEET`**: leetspeak con significado real dentro de este dataset
  (`b4` → `before`, `h8` → `hate`).

La expansión se hace con una **búsqueda en cadena**
`CONTRACTIONS.get(w, SLANG.get(w, LEET.get(w, w)))` y **no** con un mapeo
genérico de caracteres: un mapeo `4 → a` rompería formas legítimas como
`2day` → `zday`, `1st` → `ist` o `ps3` → `pse`. El leetspeak real del dataset es
escaso (≈ 0.31 % de los textos, medido sobre la muestra) y se cubre con un
diccionario curado de las formas observadas en lugar de reglas generales que
introducirían ruido. La mejora v3+v4 del pipeline cambia el resultado de
aproximadamente el 20 % de los textos (medido sobre una muestra de 10k).


In [2]:
# --- Expansión de contracciones ---
CONTRACTIONS = {
    "don't": "do not", "dont": "do not", "can't": "cannot", "cant": "cannot",
    "won't": "will not", "wont": "will not", "isn't": "is not", "isnt": "is not",
    "aren't": "are not", "arent": "are not", "didn't": "did not", "didnt": "did not",
    "doesn't": "does not", "doesnt": "does not", "wasn't": "was not", "wasnt": "was not",
    "weren't": "were not", "werent": "were not", "haven't": "have not", "havent": "have not",
    "hasn't": "has not", "hasnt": "has not", "hadn't": "had not", "hadnt": "had not",
    "i'm": "i am", "i've": "i have", "i'll": "i will", "you're": "you are",
    "you've": "you have", "we're": "we are", "they're": "they are", "it's": "it is",
    "that's": "that is", "what's": "what is", "there's": "there is",
}

# Palabras de negación que disparan el marking (not_good)

# Slang de redes sociales -> forma canónica (u = you, idk = i do not know...)
SLANG = {
    # Pronombres/verbos abreviados
    "u": "you", "ur": "your", "r": "are", "n": "and", "y": "why",
    "k": "ok", "ya": "you", "yall": "you all",
    # Contracciones coloquiales
    "gonna": "going to", "wanna": "want to", "gotta": "got to",
    "lemme": "let me", "gimme": "give me", "imma": "i am going to",
    "kinda": "kind of", "sorta": "sort of", "dunno": "do not know",
    # Abreviaturas de opinion (alimentan la negacion: idk -> i do not know)
    "idk": "i do not know", "idc": "i do not care", "ngl": "not going to lie",
    "tbh": "to be honest", "imo": "in my opinion", "imho": "in my humble opinion",
    "smh": "shake my head", "btw": "by the way", "rn": "right now",
    "fr": "for real", "af": "as fuck", "afk": "away from keyboard",
    "bc": "because", "cuz": "because", "bcoz": "because", "cos": "because",
    "tho": "though", "pls": "please", "plz": "please", "thx": "thanks",
    "wut": "what", "wat": "what", "sup": "what is up",
    "lol": "laugh out loud", "omg": "oh my god", "brb": "be right back",
    # Chatspeak fonético: variantes escritas como suenan
    "aint": "are not",           # alimenta la negación: aint funny -> not_funny
    "da": "the", "dat": "that", "dis": "this", "dem": "them",
    "dese": "these", "dose": "those", "dey": "they", "dere": "there",
    "kno": "know", "luv": "love", "gud": "good", "wuz": "was", "wus": "was",
    "pic": "picture", "pics": "pictures", "cmon": "come on", "hbu": "how about you",
}

# Leetspeak real del dataset (b4 = before, h8 = hate)
LEET = {
    # Frecuentes en este dataset (medido sobre 81k textos)
    "b4": "before", "2day": "today", "2nite": "tonight", "2night": "tonight",
    "2mrrw": "tomorrow", "2morrow": "tomorrow", "2mrw": "tomorrow", "2morw": "tomorrow",
    "str8": "straight", "str8p": "straight",
    "some1": "someone", "sum1": "someone", "any1": "anyone", "no1": "no one", "every1": "everyone",
    "h8": "hate", "4got": "forgot", "gr8": "great", "in2": "into",
    "2getha": "together", "2gether": "together", "2do": "to do", "2b": "to be",
    "4ever": "forever", "2much": "too much", "4da": "for the",
    "l8r": "later", "m8": "mate", "w8": "wait", "4u": "for you", "2u": "to you",
    "2go": "to go", "2me": "to me", "4me": "for me", "2know": "to know",
    "2see": "to see", "2have": "to have", "2make": "to make", "2give": "to give",
    "2stop": "to stop", "2come": "to come", "2show": "to show", "2bully": "to bully",
    # Insultos ofuscados (críticos para cyberbullying)
    "n00b": "noob", "bl00d": "blood", "bl00dy": "bloody", "br00t4l": "brutal",
    "r4pe": "rape", "k1ll3d": "killed", "sh00ter": "shooter", "f4ggots": "faggots",
    "b1tch": "bitch", "h4te": "hate", "l0ve": "love", "f4il": "fail", "s0": "so",
}

NEGATION_WORDS = {'no', 'not', 'never', 'nobody', 'nothing', 'nowhere', 'neither', 'nor', 'cannot'}


# --- Limpieza regex: solo caracteres, sin NLP ---
def clean(text: str) -> str:
    text = text.lower()
    text = html.unescape(text)
    text = ' '.join(CONTRACTIONS.get(word, SLANG.get(word, LEET.get(word, word))) for word in text.split())
    text = re.sub(r"[^\x00-\x7F]+", "", text)         # no ASCII
    text = re.sub(r"[a-z0-9._+-]+@[a-z0-9._+-]+\.[a-z]+", "", text)  # emails
    text = re.sub(r"http\S+", "", text)               # URLs
    text = re.sub(r"@\S+", "", text)                  # menciones
    text = re.sub(r"#\S+", "", text)                  # hashtags
    text = re.sub(r"\brt\b", " ", text)              # RT (word boundary)
    text = re.sub(r"[^a-zA-Z]", " ", text)             # solo letras
    return ' '.join(unidecode(word) for word in text.split())


def normalize_elongation(word: str) -> str:
    """Contrae la elongación: sooooo -> so, noooo -> no."""
    return re.sub(r'(.)\1{2,}', r'\1', word)


def keep_token(word) -> bool:
    """Mantiene un token si es útil. Las negaciones se conservan SIEMPRE,
    aunque sean stopwords (not, no, never...)."""
    if word.text in NEGATION_WORDS:
        return True
    return (not (word.is_punct or word.is_stop or word.like_num)
            and len(word) > 2
            and len(set(word.text)) > 1)


def mark_negation(tokens):
    """Prefija not_ al token siguiente a una negación. not good -> not_good."""
    out = []
    negated = False
    for word in tokens:
        if word in NEGATION_WORDS:
            negated = True
            continue
        out.append(('not_' if negated else '') + word)
        negated = False
    if negated:
        out.append('not')
    return out


def normalize(text: str) -> str:
    """Limpia, filtra y lematiza un texto."""
    cleaned = clean(text)
    tokens = [normalize_elongation(token.text) for token in tokenizer(cleaned) if keep_token(token)]
    lemmas = [token.lemma_ for token in lemmatizer(' '.join(tokens))]
    return ' '.join(mark_negation(lemmas))


def normalize_batch(texts, batch_size=1000, show_progress=True):
    """Versión batch de normalize(). Equivalente, ~4.5x más rápida."""
    cleaned = [clean(text) for text in texts]
    filtered = [' '.join(normalize_elongation(token.text) for token in doc if keep_token(token))
                for doc in tqdm(tokenizer.pipe(cleaned, batch_size=batch_size),
                                total=len(cleaned),
                                disable=not show_progress)]
    return [' '.join(mark_negation(token.lemma_ for token in doc))
            for doc in tqdm(lemmatizer.pipe(filtered, batch_size=batch_size),
                            total=len(filtered),
                            disable=not show_progress)]

### Prueba de humo de `normalize()`

La celda siguiente recorre un conjunto de textos de ejemplo diseñado para
verificar cada paso del pipeline:

- **Limpieza de ruido**: retuits (`RT`), URLs, menciones, hashtags, emojis y
  texto ofuscado (zalgo) deben desaparecer.
- **Negación preservada**: frases como `I am not good at this`, `don't talk to
  me` o `I can't believe it` deben conservar la negación (`not_good`, `not_talk`).
- **Elongación normalizada**: `sooooo` → `so`, `noooo` → `no`, `hiiiii` → `hi`.


### Interpretación de la prueba de humo

La prueba confirma que las decisiones de normalización operan sobre el texto
real: las negaciones se conservan y se marcan con el prefijo `not_`
(`I am not good` → `not_good`, `this is not funny` → `not_funny`, `don't
talk` → `not_talk`), la elongación de vocales se colapsa (`noooo` → `no`) y
el ruido (RT, URLs, menciones, emojis, zalgo) se elimina. El marcado de
negación es la decisión más delicada del pipeline: al anteponer `not_` al
lema, la información de la negación sobrevive a la lematización y queda
disponible para los modelos, en lugar de perderse como una stopword más.

In [3]:
sentences = [
    "RT I'm learning Python and      I'm enjoying it. :) >.<",
    "I have a website at https://www.example.com with discounts.",
    "What do you think about the new product from @company? #opinions",
    "RT @user: Thanks for the retweet. Great article! 😃",
    "10 ways to improve your mental health. #health #wellness 🧘",
    "I'm 25 years old. I was born in 1995.",
    "b̴̢̛̙̤̮͚̘̹̫̦̭̮̺̰͖͐̋̄̅͊͂̑̓̉̇̎͘̕ͅu̷̢͚̠̳̒͌͆̽̕̕ę̵̣̟̺͇̙͓͍̭̲̘̱̗͉̆̃̇͜͜ṅ̵̢̥̱̳̙̫̩̘̪̣̬͓͇̮̕͜ȁ̵̙͖̗̓̆́̊́̈́̕s̸̛͉͉͓̮͌͜",
    "zzzz", "Hahaha", "eye",
    "I am not good at this",
    "this is not funny",
    "don't talk to me",
    "I can't believe it",
    "I do not like you",
    "sooooo good",
    "noooo",
    "yesssss",
    "hiiiii how are you",
]

for original, normalized in zip(sentences, normalize_batch(sentences, show_progress=False)):
    print("TXT:", original)
    print("OUT:", normalized)
    print()

TXT: RT I'm learning Python and      I'm enjoying it. :) >.<
OUT: learn python enjoy

TXT: I have a website at https://www.example.com with discounts.
OUT: website discount

TXT: What do you think about the new product from @company? #opinions
OUT: think new product

TXT: RT @user: Thanks for the retweet. Great article! 😃
OUT: thank retweet great article

TXT: 10 ways to improve your mental health. #health #wellness 🧘
OUT: way improve mental health

TXT: I'm 25 years old. I was born in 1995.
OUT: year old bear

TXT: b̴̢̛̙̤̮͚̘̹̫̦̭̮̺̰͖͐̋̄̅͊͂̑̓̉̇̎͘̕ͅu̷̢͚̠̳̒͌͆̽̕̕ę̵̣̟̺͇̙͓͍̭̲̘̱̗͉̆̃̇͜͜ṅ̵̢̥̱̳̙̫̩̘̪̣̬͓͇̮̕͜ȁ̵̙͖̗̓̆́̊́̈́̕s̸̛͉͉͓̮͌͜
OUT: buena

TXT: zzzz
OUT: 

TXT: Hahaha
OUT: hahaha

TXT: eye
OUT: eye

TXT: I am not good at this
OUT: not_good

TXT: this is not funny
OUT: not_funny

TXT: don't talk to me
OUT: not_talk

TXT: I can't believe it
OUT: not_believe

TXT: I do not like you
OUT: not_like

TXT: sooooo good
OUT: so good

TXT: noooo
OUT: not

TXT: yesssss
OUT: yes

TXT: hiiiii how are you
OU

## Preprocesamiento de un dataset

La celda siguiente aplica `normalize_batch()` al dataset crudo completo
(`../data/raw/cyberbullying.csv`) y genera el dataset intermedio que consumen
las etapas siguientes. Los pasos, en orden:

1. **Carga del dataset crudo** y renombrado de columnas a `text` / `label`.
2. **Normalización en lote** con `normalize_batch()`: la versión batch usa
   `tokenizer.pipe()` y `lemmatizer.pipe()` para procesar los textos por lotes, lo que es
   ~4.5 veces más rápido que el loop de una sola llamada.
3. **Eliminación de filas vacías**: los textos que quedan vacíos tras normalizar
   (sin tokens útiles) no aportan señal y se descartan.
4. **Guardado** en `../data/processed/cyberbullying_preprocessed.csv`
   (columnas `text`, `label`, `text_preprocessed`).


In [4]:
import os

renormalize = False  # False: reutiliza el CSV ya generado; True: fuerza re-normalizar todo.
output_path = '../data/processed/cyberbullying_preprocessed.csv'

if not renormalize and os.path.exists(output_path):
    df = pd.read_csv(output_path)
    print('Normalizacion ya existente. Cargada desde:', output_path)
    print('Para re-normalizar desde cero, pone renormalize = True en esta celda y volve a correrla.')
else:
    df = pd.read_csv('../data/raw/cyberbullying.csv')
    df.columns = ['text', 'label']
    df['text_preprocessed'] = normalize_batch(df['text'])
    df = df[df['text_preprocessed'] != '']
    df.to_csv(output_path, index=False)
    print('Normalizacion completada y guardada en:', output_path)

print('Filas en df:', len(df))
df.head()


100%|██████████| 81417/81417 [00:45<00:00, 1783.59it/s]


,text,label,text_preprocessed
0,"In other words #katandandre, your food was cra...",0,word food crapilicious
1,Why is #aussietv so white? #MKR #theblock #ImA...,0,white
2,@XochitlSuckkks a classy whore? Or more red ve...,0,classy whore red velvet cupcake
3,"@Jason_Gio meh. :P thanks for the heads up, b...",0,meh thank head not_concerned angry dude twitter
4,@RudhoeEnglish This is an ISIS account pretend...,0,isis account pretend kurdish account like isla...


### Recarga desde el CSV en disco

La celda siguiente recarga el CSV ya generado para inspeccionarlo; **no** depende
de la variable `df`. Según el interruptor `renormalize` de la celda de
normalización, los conteos pueden diferir:

- Con `renormalize = False` (por defecto): la celda de normalización **recarga**
  el CSV ya existente (80.909 registros) y no re-procesa el crudo.
- Con `renormalize = True`: re-procesa las **81.417 filas** del dataset crudo y
  regenera el CSV.

El `head()` se ve igual pero el total de filas depende de la fuente.


### Interpretación: tamaño del dataset resultante

El pipeline procesa 81 417 filas crudas y produce **80 909 registros**
(`cyberbullying_preprocessed.csv`): las filas cuyo texto queda vacío tras la
normalización se descartan (`drop_empty_text`), y el dataset intermedio es el
que consumen las etapas posteriores (EDA, features y modelado). La diferencia
entre 81 417 y 80 909 (~0,6 %) muestra que el preprocesado no degrada
significativamente la cobertura del corpus.

In [5]:
df = pd.read_csv('../data/processed/cyberbullying_preprocessed.csv')
df.head()

,text,label,text_preprocessed
0,"In other words #katandandre, your food was cra...",0,word food crapilicious
1,Why is #aussietv so white? #MKR #theblock #ImA...,0,white
2,@XochitlSuckkks a classy whore? Or more red ve...,0,classy whore red velvet cupcake
3,"@Jason_Gio meh. :P thanks for the heads up, b...",0,meh thank head not_concerned angry dude twitter
4,@RudhoeEnglish This is an ISIS account pretend...,0,isis account pretend kurdish account like isla...


## Decisiones de normalización y justificación

Resumen de las decisiones tomadas en esta etapa, a efectos de la metodología:

- **Minúsculas y Unicode (`unidecode`)**: se unifican variantes de acentuación y
  mayúsculas para reducir la dispersión del vocabulario.
- **Entidades HTML (`html.unescape`)**: se decodifican antes de la limpieza regex
  para que `&amp;` no contamine el vocabulario.
- **Expansión de contracciones, slang y leetspeak**: se normalizan las formas
  coloquiales de redes y se **preserva la negación** (`don't` → `do not`), señal
  crítica para cyberbullying. El leetspeak se cubre con un diccionario curado de
  las formas observadas (≈ 0.31 % del dataset) en lugar de reglas generales.
- **Eliminación de no-ASCII, emails, URLs, menciones, hashtags y `RT`**: son
  ruido propio de la plataforma Twitter que no aporta señal de cyberbullying y,
  en el caso de URLs y menciones, introduce tokens no generalizables a texto
  nuevo.
- **Solo letras `a-zA-Z` y filtrado spaCy** (stopwords/puntuación/números,
  longitud > 2, caracteres no repetidos): se eliminan tokens sin contenido
  semántico y ruido tipo `zzzz`.
- **Elongación**: se contraen las repeticiones tipográficas (`sooooo` → `so`)
  para que las variantes de una misma palabra compartan token.
- **Lematización con `lemmatizer` completo y marcado de negación sobre los lemas**: el
  contexto del parser mejora el lema; marcar la negación después evita que el
  prefijo `not_` rompa la lematización (`not_going` → `not_goe`).
- **Doble pipeline de spaCy**: `tokenizer` (mínimo) para filtrar ~8x más rápido y
  `lemmatizer` (completo) solo para lematizar.
- **Eliminación de filas vacías**: los textos que quedan vacíos tras normalizar
  se descartan del dataset.

El resultado de esta etapa es el dataset
`../data/processed/cyberbullying_preprocessed.csv` (columnas `text`,
`label`, `text_preprocessed`), que alimenta las etapas de características (01),
análisis exploratorio (02) y modelado (03).
